<a href="https://colab.research.google.com/github/Caio-R-C/RemoteSensingStudies/blob/main/visualizacao_3d_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualização 3D Interativa e Movível (Plotly + VTK)
### Mestrado - Fazenda Água Limpa (FAL) | Investigação de Aquífero Fraturado

Este notebook processa o modelo tridimensional do ResIPy (`3Dfrom2Dlines_1.5.vtk`) e renderiza cenas **totalmente interativas e movíveis em 3D via Plotly (WebGL)**:

* 🔄 **Girar o modelo:** Clique com o botão esquerdo e arraste em qualquer direção.
* 🔍 **Zoom contínuo:** Use o scroll do mouse ou pinça do touchpad.
* ✋ **Transladar (Pan):** Clique com o botão direito (ou Shift + clique esquerdo) e arraste.
* 👁️ **Ligar / Desligar camadas:** Clique nos itens da legenda (à direita) para ocultar ou isolar seções, perfis e isosuperfícies.
* 📍 **Inspeção de valores (Hover):** Passe o mouse sobre qualquer ponto para ler as coordenadas UTM reais $(X, Y, Z)$ e a resistividade em $\Omega\cdot	ext{m}$.
* 🌐 **Exportação HTML Standalone:** Cada figura gera automaticamente um arquivo `.html` interativo que abre em qualquer navegador sem depender do Jupyter!


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv
import plotly.graph_objects as go
import plotly.io as pio

# Configura renderizador padrão do Plotly para ambiente Jupyter
pio.renderers.default = 'notebook_connected'

# Diretório dos dados da inversão Pseudo-3D
PATH_DIR = r"C:\Faculdade\MESTRADO\DADOS - FAL\PERFIS - RESISTIVIDADE\DADOS - P1, P2, P3 e P4\pseudo 3d - inversão"

# Seleção do arquivo da malha 3D (prioriza o modelo _1.5)
PATH_VTK_MESH = os.path.join(PATH_DIR, "3Dfrom2Dlines_1.5.vtk")
if not os.path.exists(PATH_VTK_MESH):
    PATH_VTK_MESH = os.path.join(PATH_DIR, "3Dfrom2Dlines.vtk")

PATH_VTK_ELEC = os.path.join(PATH_DIR, "eletrodos.vtk")

print("Verificação dos arquivos de entrada:")
print(f"  • Malha 3D (.vtk):  {'OK' if os.path.exists(PATH_VTK_MESH) else 'NÃO ENCONTRADO'} -> {PATH_VTK_MESH}")
print(f"  • Eletrodos (.vtk): {'OK' if os.path.exists(PATH_VTK_ELEC) else 'NÃO ENCONTRADO'} -> {PATH_VTK_ELEC}")


Verificação dos arquivos de entrada:
  • Malha 3D (.vtk):  OK -> C:\Faculdade\MESTRADO\DADOS - FAL\PERFIS - RESISTIVIDADE\DADOS - P1, P2, P3 e P4\pseudo 3d - inversão\3Dfrom2Dlines_1.5.vtk
  • Eletrodos (.vtk): OK -> C:\Faculdade\MESTRADO\DADOS - FAL\PERFIS - RESISTIVIDADE\DADOS - P1, P2, P3 e P4\pseudo 3d - inversão\eletrodos.vtk


In [ ]:
# Carrega dados do VTK via PyVista
mesh = pv.read(PATH_VTK_MESH)
elec = pv.read(PATH_VTK_ELEC)

# Converte dados de células para nós (pontos), permitindo gradientes suaves e isosuperfícies
mesh_pt = mesh.cell_data_to_point_data()

xmin, xmax, ymin, ymax, zmin, zmax = mesh.bounds

print("="*70)
print("  DIAGNÓSTICO ESPACIAL DO BLOCO 3D (UTM / COTA)")
print("="*70)
print(f"  • Células volumétricas: {mesh.n_cells:,}")
print(f"  • Vértices da malha:    {mesh.n_points:,}")
print(f"  • Extensão X (Leste UTM):  {xmin:,.1f} m até {xmax:,.1f} m (Largura: {xmax-xmin:.1f} m)")
print(f"  • Extensão Y (Norte UTM):  {ymin:,.1f} m até {ymax:,.1f} m (Comprimento: {ymax-ymin:.1f} m)")
print(f"  • Profundidade Z (Cota):   {zmin:,.1f} m até {zmax:,.1f} m (Investigação: {abs(zmin):.1f} m)")
print(f"  • Total de Eletrodos:     {elec.n_points}")
print("="*70)

def pv_to_plotly_mesh(pv_mesh, scalar_name=None, color=None, opacity=1.0, name='Malha 3D',
                       showscale=False, colorscale='Turbo', cmin=2.5, cmax=5.0):
    """
    Converte qualquer malha ou fatia do PyVista em um objeto Mesh3d interativo do Plotly.
    """
    tri = pv_mesh.triangulate()
    faces = tri.faces.reshape(-1, 4)[:, 1:4]

    kwargs = dict(
        x=tri.points[:, 0],
        y=tri.points[:, 1],
        z=tri.points[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        opacity=opacity,
        name=name,
        showlegend=True,
        hoverinfo='x+y+z+name'
    )

    if scalar_name and scalar_name in tri.point_data:
        vals = tri.point_data[scalar_name]
        kwargs['intensity'] = vals
        kwargs['colorscale'] = colorscale
        kwargs['cmin'] = cmin
        kwargs['cmax'] = cmax
        kwargs['showscale'] = showscale
        if showscale:
            kwargs['colorbar'] = dict(
                title='log10(Resistividade)',
                len=0.7,
                thickness=18,
                x=1.02
            )
    elif color:
        kwargs['color'] = color

    return go.Mesh3d(**kwargs)

print("✅ Função de integração PyVista -> Plotly carregada com sucesso!")


  DIAGNÓSTICO ESPACIAL DO BLOCO 3D (UTM / COTA)
  • Células volumétricas: 159,661
  • Vértices da malha:    30,825
  • Extensão X (Leste UTM):  183,884.6 m até 185,546.9 m (Largura: 1662.3 m)
  • Extensão Y (Norte UTM):  8,230,525.2 m até 8,231,771.2 m (Comprimento: 1246.0 m)
  • Profundidade Z (Cota):   -559.9 m até 0.0 m (Investigação: 559.9 m)
  • Total de Eletrodos:     224
✅ Função de integração PyVista -> Plotly carregada com sucesso!


In [ ]:
res = mesh.cell_data['Resistivity(ohm.m)']
log_res = mesh.cell_data['Resistivity(log10)']

percentis = [1, 5, 10, 25, 50, 75, 90, 95]
valores_p = np.percentile(res, percentis)

print("="*70)
print("  DISTRIBUIÇÃO DOS VALORES DE RESISTIVIDADE (3Dfrom2Dlines_1.5.vtk):")
print("="*70)
for p, v in zip(percentis, valores_p):
    print(f"    P{p:02d}: {v:10.1f} ohm.m  |  log10 = {np.log10(v):.2f}")
print("="*70)

# Limiares hidrogeológicos recomendados para as isosuperfícies:
#   800 Ω·m   (log10 = 2.90) -> Núcleo hiper-condutivo saturado
#   2.000 Ω·m (log10 = 3.30) -> Aquífero Fraturado Principal
#   6.000 Ω·m (log10 = 3.78) -> Manto de Intemperismo / Saprolito
#  20.000 Ω·m (log10 = 4.30) -> Topo da Rocha Sã (Bedrock)


  DISTRIBUIÇÃO DOS VALORES DE RESISTIVIDADE (3Dfrom2Dlines_1.5.vtk):
    P01:      693.9 ohm.m  |  log10 = 2.84
    P05:     1569.1 ohm.m  |  log10 = 3.20
    P10:     2600.8 ohm.m  |  log10 = 3.42
    P25:     7122.7 ohm.m  |  log10 = 3.85
    P50:    17919.9 ohm.m  |  log10 = 4.25
    P75:    33754.0 ohm.m  |  log10 = 4.53
    P90:    89454.8 ohm.m  |  log10 = 4.95
    P95:   116726.7 ohm.m  |  log10 = 5.07


In [ ]:
# ==============================================================================
# 1. MODELO DE CAMADAS HORIZONTAIS EMPILHADAS (PLOTLY 3D INTERATIVO)
# Movível, rotacionável com o mouse e com controle de visibilidade na legenda!
# ==============================================================================
profundidades = [-5, -15, -25, -40, -60, -85, -115, -150]
fig_camadas = go.Figure()

print("Gerando fatias horizontais interativas...")
for idx, z_prof in enumerate(profundidades):
    sl = mesh_pt.slice(normal='z', origin=(0, 0, z_prof))
    if sl.n_cells > 0:
        # Exibe a barra de escala de cores apenas na primeira fatia
        show_scale = (idx == 0)
        trace_slice = pv_to_plotly_mesh(
            sl,
            scalar_name='Resistivity(log10)',
            opacity=0.92,
            name=f'Fatia Z = {z_prof} m',
            showscale=show_scale,
            colorscale='Turbo',
            cmin=2.5,
            cmax=5.0
        )
        fig_camadas.add_trace(trace_slice)

# Adiciona os eletrodos das 4 linhas na superfície
trace_elec = go.Scatter3d(
    x=elec.points[:, 0],
    y=elec.points[:, 1],
    z=elec.points[:, 2] + 2.0,  # leve elevação para não sumir no relevo
    mode='markers',
    marker=dict(size=3, color='black'),
    name='Eletrodos P1-P4',
    hovertext=[f'Eletrodo {i+1}' for i in range(len(elec.points))],
    hoverinfo='text+x+y+z'
)
fig_camadas.add_trace(trace_elec)

# Configuração de layout 3D (com exagero vertical z=0.45 para melhor proporção)
fig_camadas.update_layout(
    title=dict(
        text='<b>Modelo de Camadas Horizontais Empilhadas (Interativo 3D - FAL)</b><br><sup>Gire com o mouse | Clique na legenda para ligar/desligar níveis de profundidade</sup>',
        font=dict(size=15)
    ),
    scene=dict(
        xaxis=dict(title='UTM Leste (m)', showgrid=True),
        yaxis=dict(title='UTM Norte (m)', showgrid=True),
        zaxis=dict(title='Cota Z (m)', showgrid=True),
        aspectmode='manual',
        aspectratio=dict(x=1.0, y=1.0, z=0.45),
        camera=dict(
            eye=dict(x=-1.5, y=-1.5, z=0.9)
        )
    ),
    width=1000,
    height=750,
    margin=dict(l=10, r=10, t=60, b=10)
)

# Salva arquivo HTML standalone (abre interativo no navegador!)
html_camadas = os.path.join(PATH_DIR, "modelo_camadas_empilhadas_3d.html")
fig_camadas.write_html(html_camadas)
print(f"✅ Arquivo HTML interativo salvo em: {html_camadas}")

fig_camadas.show()


Gerando fatias horizontais interativas...
✅ Arquivo HTML interativo salvo em: C:\Faculdade\MESTRADO\DADOS - FAL\PERFIS - RESISTIVIDADE\DADOS - P1, P2, P3 e P4\pseudo 3d - inversão\modelo_camadas_empilhadas_3d.html


In [ ]:
# ==============================================================================
# 2. PAINEL DE CERCAS (FENCE DIAGRAM) COM ISOSUPERFÍCIE (PLOTLY 3D INTERATIVO)
# Similar à figura da direita da literatura: seções verticais dos perfis + corpo 3D
# ==============================================================================

fig_cercas = go.Figure()
pts = elec.points

print("Calculando seções verticais das linhas P1 a P4...")
nomes_linhas = ['Linha P1', 'Linha P2', 'Linha P3', 'Linha P4']

for i in range(4):
    p1 = pts[i * 56]
    p2 = pts[(i + 1) * 56 - 1]
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    normal = (-dy, dx, 0)
    origin = (p1 + p2) / 2
    sl = mesh_pt.slice(normal=normal, origin=origin)

    trace_perfil = pv_to_plotly_mesh(
        sl,
        scalar_name='Resistivity(log10)',
        opacity=0.90,
        name=nomes_linhas[i],
        showscale=(i == 0),
        colorscale='Turbo',
        cmin=2.5,
        cmax=5.0
    )
    fig_cercas.add_trace(trace_perfil)

# Cálculo da Isosuperfície 3D do aquífero fraturado (<= 2.000 ohm.m)
LIMIAR_ISO_OHM = 2000.0
log_iso = np.log10(LIMIAR_ISO_OHM)

print(f"Calculando Isosuperfície 3D em {LIMIAR_ISO_OHM:.0f} ohm.m...")
iso_aquifero = mesh_pt.contour(isosurfaces=[log_iso], scalars='Resistivity(log10)')

trace_iso = pv_to_plotly_mesh(
    iso_aquifero,
    color='#d62728',  # Vermelho
    opacity=0.60,
    name=f'Aquífero Fraturado ({LIMIAR_ISO_OHM:.0f} Ω·m)'
)
fig_cercas.add_trace(trace_iso)

# Eletrodos na superfície
fig_cercas.add_trace(trace_elec)

fig_cercas.update_layout(
    title=dict(
        text=f'<b>Painel de Cercas dos Perfis + Isosuperfície 3D ({LIMIAR_ISO_OHM:.0f} Ω·m)</b><br><sup>Interativo via Plotly | Gire livremente e inspecione o cruzamento dos corpos</sup>',
        font=dict(size=15)
    ),
    scene=dict(
        xaxis=dict(title='UTM Leste (m)'),
        yaxis=dict(title='UTM Norte (m)'),
        zaxis=dict(title='Cota Z (m)'),
        aspectmode='manual',
        aspectratio=dict(x=1.0, y=1.0, z=0.45),
        camera=dict(
            eye=dict(x=-1.6, y=-1.4, z=0.8)
        )
    ),
    width=1000,
    height=750,
    margin=dict(l=10, r=10, t=60, b=10)
)

html_cercas = os.path.join(PATH_DIR, "painel_cercas_isosuperficie_3d.html")
fig_cercas.write_html(html_cercas)
print(f"✅ Arquivo HTML interativo salvo em: {html_cercas}")



fig_cercas.show()


Calculando seções verticais das linhas P1 a P4...
Calculando Isosuperfície 3D em 2000 ohm.m...
✅ Arquivo HTML interativo salvo em: C:\Faculdade\MESTRADO\DADOS - FAL\PERFIS - RESISTIVIDADE\DADOS - P1, P2, P3 e P4\pseudo 3d - inversão\painel_cercas_isosuperficie_3d.html


In [ ]:
# ==============================================================================
# 3. MÚLTIPLAS ISOSUPERFÍCIES ANINHADAS INTERATIVAS (PLOTLY 3D)
# Cascas hidrogeológicas: clique na legenda para ligar/desligar e ver por dentro!
# ==============================================================================

print("Calculando as 3 isosuperfícies concêntricas...")
iso_800  = mesh_pt.contour(isosurfaces=[np.log10(800.0)], scalars='Resistivity(log10)')
iso_2000 = mesh_pt.contour(isosurfaces=[np.log10(2000.0)], scalars='Resistivity(log10)')
iso_20k  = mesh_pt.contour(isosurfaces=[np.log10(20000.0)], scalars='Resistivity(log10)')

fig_multi = go.Figure()

# Topo da Rocha Sã (Bedrock) - Cinza claro translúcido
fig_multi.add_trace(pv_to_plotly_mesh(
    iso_20k,
    color='#7f7f7f',
    opacity=0.22,
    name='Topo Rocha Sã (20.000 Ω·m)'
))

# Aquífero Fraturado Principal - Laranja
fig_multi.add_trace(pv_to_plotly_mesh(
    iso_2000,
    color='#ff7f0e',
    opacity=0.55,
    name='Aquífero Fraturado (2.000 Ω·m)'
))

# Núcleo Hipersaturado / Fratura Condutiva - Azul vivo
fig_multi.add_trace(pv_to_plotly_mesh(
    iso_800,
    color='#1f77b4',
    opacity=0.85,
    name='Núcleo Saturado (800 Ω·m)'
))

# Eletrodos na superfície
fig_multi.add_trace(trace_elec)

fig_multi.update_layout(
    title=dict(
        text='<b>Múltiplas Isosuperfícies Hidrogeológicas 3D (Plotly)</b><br><sup>Dica: Clique em "Topo Rocha Sã" na legenda para ocultar a casca externa e ver o aquífero interno!</sup>',
        font=dict(size=14)
    ),
    scene=dict(
        xaxis=dict(title='UTM Leste (m)'),
        yaxis=dict(title='UTM Norte (m)'),
        zaxis=dict(title='Cota Z (m)'),
        aspectmode='manual',
        aspectratio=dict(x=1.0, y=1.0, z=0.45),
        camera=dict(
            eye=dict(x=-1.5, y=-1.5, z=0.9)
        )
    ),
    width=1000,
    height=750,
    margin=dict(l=10, r=10, t=60, b=10)
)

html_multi = os.path.join(PATH_DIR, "multiplas_isosuperficies_3d.html")
fig_multi.write_html(html_multi)
print(f"✅ Arquivo HTML interativo salvo em: {html_multi}")

fig_multi.show()


Calculando as 3 isosuperfícies concêntricas...
✅ Arquivo HTML interativo salvo em: C:\Faculdade\MESTRADO\DADOS - FAL\PERFIS - RESISTIVIDADE\DADOS - P1, P2, P3 e P4\pseudo 3d - inversão\multiplas_isosuperficies_3d.html


In [ ]:
# ==============================================================================
# 4. ISOLAMENTO VOLUMÉTRICO E CÁLCULO NUMÉRICO DO VOLUME DO AQUÍFERO (m³)
# ==============================================================================

LIMIAR_CORTE_OHM = 2000.0
log_corte = np.log10(LIMIAR_CORTE_OHM)

# Filtro de células volumétricas dentro do domínio aquífero
aquifero_solido = mesh.threshold(
    value=[mesh.cell_data['Resistivity(log10)'].min(), log_corte],
    scalars='Resistivity(log10)'
)

vol_aquifero_m3 = aquifero_solido.volume
vol_total_m3 = mesh.volume
pct_vol = (vol_aquifero_m3 / vol_total_m3) * 100.0

porosidades = [0.01, 0.02, 0.03]

print("="*70)
print("  ESTIMATIVA VOLUMÉTRICA DO RESERVATÓRIO FRATURADO (FAL)")
print("="*70)
print(f"  • Células no domínio aquífero (<= {LIMIAR_CORTE_OHM:.0f} ohm.m): {aquifero_solido.n_cells:,} de {mesh.n_cells:,}")
print(f"  • Volume geométrico do aquífero:  {vol_aquifero_m3:,.0f} m³ ({pct_vol:.1f}% do volume investigado)")
print(f"  • Volume total da malha 3D:       {vol_total_m3:,.0f} m³")
print("-"*70)
print("  ESTIMATIVAS DE RESERVA HÍDRICA ARMAZENADA (VOLUME DE ÁGUA):")
for phi in porosidades:
    vol_agua_m3 = vol_aquifero_m3 * phi
    print(f"    - Porosidade {phi*100:.0f}%: {vol_agua_m3:,.0f} m³ ({vol_agua_m3/1e3:,.1f} milhões de litros)")
print("="*70)


  ESTIMATIVA VOLUMÉTRICA DO RESERVATÓRIO FRATURADO (FAL)
  • Células no domínio aquífero (<= 2000 ohm.m): 11,914 de 159,661
  • Volume geométrico do aquífero:  4,480,476 m³ (0.9% do volume investigado)
  • Volume total da malha 3D:       475,933,709 m³
----------------------------------------------------------------------
  ESTIMATIVAS DE RESERVA HÍDRICA ARMAZENADA (VOLUME DE ÁGUA):
    - Porosidade 1%: 44,805 m³ (44.8 milhões de litros)
    - Porosidade 2%: 89,610 m³ (89.6 milhões de litros)
    - Porosidade 3%: 134,414 m³ (134.4 milhões de litros)


In [ ]:
# ==============================================================================
# 5. EXPORTAÇÃO TABULAR (CSV) PARA SIG / QGIS / CORRELAÇÃO COM POÇOS
# ==============================================================================

print("Extraindo centroides e propriedades de cada célula da malha 3D...")
centros = mesh.cell_centers().points

df_3d = pd.DataFrame({
    'X_UTM': centros[:, 0],
    'Y_UTM': centros[:, 1],
    'Z_Cota': centros[:, 2],
    'Resistividade_ohm_m': mesh.cell_data['Resistivity(ohm.m)'],
    'Resistividade_log10': mesh.cell_data['Resistivity(log10)'],
    'Sensibilidade_log10': mesh.cell_data['Sensitivity_map(log10)']
})

arquivo_csv = os.path.join(PATH_DIR, "pseudo3d_resultados_celulas.csv")
df_3d.to_csv(arquivo_csv, index=False)

print(f"✅ Tabela CSV gerada com sucesso!")
print(f"   Arquivo: {arquivo_csv}")
print(f"   Total de registros: {len(df_3d):,}")
display(df_3d.head(10))
